# Silver Processing — Brands

Cleans `ecommerce.bronze.bronze_brands` into `ecommerce.silver.silver_brands`:

* **All business columns** (`brand_code`, `brand_name`, `category_code`): trim extra spaces and strip trailing special characters
* **`category_code` standardization** — one canonical code per category: `TOY` (toys), `BKS` (books), `GRCY` (grocery), plus `CE`, `APP`, `HNK`, `BPC`, `SPT`
* **Metadata columns** (`source_file`, `ingested_at`) pass through untouched

In [0]:
from pyspark.sql.functions import col, trim, upper, when
from pyspark.sql.functions import regexp_replace

bronze_table = "ecommerce.bronze.bronze_brands"
target_table = "ecommerce.silver.silver_brands"

df_bronze_brands = spark.table(bronze_table)

# Clean a business column: trim spaces and strip any trailing special characters
def clean(column_name):
    return regexp_replace(trim(col(column_name)), "[^A-Za-z0-9]+$", "")

df_silver_brands = (
    df_bronze_brands
    .withColumn("brand_code", clean("brand_code"))
    .withColumn("brand_name", clean("brand_name"))
    # Standardize category_code: trim + uppercase, then map variants to one canonical code per category
    .withColumn(
        "category_code",
        when(clean("category_code").isin("TOY", "TOYS"), "TOY")
        .when(clean("category_code").isin("BKS", "BOOKS"), "BKS")
        .when(clean("category_code").isin("GRCY", "GROCERY"), "GRCY")
        .otherwise(upper(clean("category_code"))),
    )
)

# Idempotent full load into the silver table
(
    df_silver_brands.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.silver.silver_brands"))

In [0]:
# Check 1: category_code should have exactly one code per category (8 canonical codes)
spark.sql("""
    SELECT category_code, COUNT(*) AS cnt
    FROM ecommerce.silver.silver_brands
    GROUP BY category_code
    ORDER BY category_code
""").show()

# Check 2: no residual spaces or special characters in business columns
spark.sql("""
    SELECT COUNT(*) AS dirty_rows
    FROM ecommerce.silver.silver_brands
    WHERE brand_code != trim(brand_code)
       OR brand_code RLIKE '[^A-Za-z0-9]$'
       OR brand_name != trim(brand_name)
       OR brand_name RLIKE '[^A-Za-z0-9]$'
       OR category_code RLIKE '[^A-Za-z0-9]$'
""").show()